# 02 · Nested configuration selection (Table II)

For every outer fold the held-out cell is removed first. Each model's four candidate
configurations (cycle age yes/no x target RUL or RUL/k) are scored by leave-one-cell-out
over the nine remaining cells; the winner is refit on the nine cells and applied once to
the held-out cell. The held-out cell never influences any choice. The four fixed
configurations are also scored, as an ablation only.

In [1]:
%run ./00_common.ipynb

root: <repository root>


In [2]:
cells = load_cells()
scr = screen(cells, save=False)
elig = scr[scr["eligible"]]["battery"].tolist()
df, cols = feature_table(cells, elig, "trailing")
df["age"] = df["cycle"].astype(float)
models = list(get_models().keys())

In [3]:
t0 = time.time()
nested, fixed, choices = {}, {c: {} for c in CONFIGS}, []
for held in elig:
    train = df[df["battery_id"] != held]
    test = df[df["battery_id"] == held].sort_values("cycle")
    inner_cells = sorted(train["battery_id"].unique())
    for name in models:
        score = {}
        for cfg, (age, ratio) in CONFIGS.items():
            errs_in = []
            for j in inner_cells:
                tr_j, va_j = train[train.battery_id != j], train[train.battery_id == j].sort_values("cycle")
                errs_in.append(cell_err(cells, j, va_j["cycle"].to_numpy(), fit_predict(name, tr_j, va_j, cols, age, ratio)))
            score[cfg] = float(np.mean(errs_in))
            fixed[cfg].setdefault(name, {})[held] = dict(cycle=test["cycle"].to_numpy(),
                rul_pred=fit_predict(name, train, test, cols, age, ratio), rul_true=test["rul"].to_numpy(float))
        best = min(score, key=score.get)
        choices.append(dict(held_out=held, model=name, chosen=best, **{f"inner_{c}": round(v, 2) for c, v in score.items()}))
        nested.setdefault(name, {})[held] = fixed[best][name][held]
    print(f"outer fold {held} done ({time.time() - t0:.0f}s)")
ch = pd.DataFrame(choices)
ch.to_csv(RESULTS / "nested_choices.csv", index=False)
ch.groupby(["model", "chosen"]).size().unstack(fill_value=0)

outer fold B0005 done (38s)


outer fold B0006 done (77s)


outer fold B0007 done (116s)


outer fold B0018 done (152s)


outer fold B0036 done (190s)


outer fold B0042 done (229s)


outer fold B0043 done (267s)


outer fold B0044 done (309s)


outer fold B0054 done (348s)


outer fold B0055 done (383s)


chosen,age,age_ratio,base
model,,,
LightGBM,3,1,6
Linear,0,0,10
RandomForest,0,10,0
Ridge,0,0,10
XGBoost,0,9,1


In [4]:
e_n = per_cell_errors(nested, cells); e_n["config"] = "nested"
parts = [e_n]
for cfg, P in fixed.items():
    e = per_cell_errors(P, cells); e["config"] = f"fixed_{cfg}"; parts.append(e)
errs = pd.concat(parts, ignore_index=True)
errs.to_csv(RESULTS / "nested_per_cell_errors.csv", index=False)
summ = pd.concat([summarise(errs[errs.config == c], c) for c in errs.config.unique()])
summ.to_csv(RESULTS / "nested_summary.csv", index=False)
print("Table II (nested):")
summ[summ.config == "nested"][["method", "lambda", "MAE", "MAE_sd", "MAE_ci_lo", "MAE_ci_hi", "RMSE", "HR", "alpha_acc"]].round(2)

Table II (nested):


,method,lambda,MAE,MAE_sd,MAE_ci_lo,MAE_ci_hi,RMSE,HR,alpha_acc
0,LightGBM,0.5,20.23,24.03,7.90,35.70,30.48,0.4,0.3
1,LightGBM,0.7,12.22,10.57,6.42,18.73,15.80,0.6,0.2
2,LightGBM,0.9,5.86,6.54,2.75,10.22,8.53,0.9,0.1
3,Linear,0.5,28.69,30.58,12.68,48.30,40.80,0.5,0.4
4,Linear,0.7,21.23,18.39,11.59,32.98,27.48,0.3,0.1
5,Linear,0.9,15.26,14.55,8.15,25.13,20.58,0.5,0.1
6,RandomForest,0.5,17.81,22.15,8.07,32.53,27.54,0.4,0.3
7,RandomForest,0.7,9.36,12.61,3.21,17.65,15.19,0.7,0.5
8,RandomForest,0.9,4.13,4.39,1.77,6.93,5.87,0.8,0.4
9,Ridge,0.5,25.04,17.08,15.73,35.56,29.83,0.1,0.1


In [5]:
w = wilcoxon_table(e_n)
w.to_csv(RESULTS / "nested_wilcoxon_l07.csv", index=False)
w.round(3)

,a,b,mean_a,mean_b,p
0,LightGBM,Linear,12.215,21.234,0.020
1,LightGBM,RandomForest,12.215,9.361,0.322
2,LightGBM,Ridge,12.215,12.361,1.000
3,LightGBM,XGBoost,12.215,9.591,0.064
4,Linear,RandomForest,21.234,9.361,0.014
5,Linear,Ridge,21.234,12.361,0.109
6,Linear,XGBoost,21.234,9.591,0.004
7,RandomForest,Ridge,9.361,12.361,0.375
8,RandomForest,XGBoost,9.361,9.591,0.232
9,Ridge,XGBoost,12.361,9.591,0.625


Ablation only (not used for selection): MAE at lambda = 0.7 of each fixed configuration.

In [6]:
summ[summ["lambda"] == 0.7].pivot(index="method", columns="config", values="MAE").round(2)

config,fixed_age,fixed_age_ratio,fixed_base,fixed_ratio,nested
method,,,,,
LightGBM,13.80,16.44,13.06,19.47,12.22
Linear,24.76,29.61,21.23,28.97,21.23
RandomForest,12.45,9.36,12.25,7.80,9.36
Ridge,17.79,31.42,12.36,21.74,12.36
XGBoost,12.99,9.55,11.82,17.05,9.59
